# Car prices — cleaning and preprocessing
The original notebook dropped Model, all missing rows, several percentile tails and engines above 6.5 L. I keep its high-cardinality Model decision and engine validity rule, while retaining missing predictor rows and valid extremes. Rows without a known positive price cannot be used for supervised training or evaluation.

Exact duplicate source records are reduced to their first occurrence before splitting. RowId is the original one-based CSV row number and never enters model fitting.

**My notes:** review missingness and invalid measurements before refining the rules.


## Setup and label/split validation


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display, HTML

# Works when Jupyter starts in this folder, notebooks/, or the repository root.
locations = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (candidate for location in locations
     for candidate in (location, location / "regression" / "car_price_prediction")
     if (candidate / "car_prices_dataset" / "train" / "car_price.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository or car_price_prediction folder.")
DATA_DIR = PROJECT_ROOT / "car_prices_dataset"
TRAIN_PATH = DATA_DIR / "train" / "car_price.csv"
GROUND_TRUTH_DIR = DATA_DIR / "ground_truth"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
REPORTS_DIR = PROJECT_ROOT / "reports"
TARGET = "Price"
ID_COLUMN = "RowId"
pio.renderers.default = "plotly_mimetype"
pd.set_option("display.max_columns", 20)


In [2]:
# Match these split/feature settings to training if you change them.
config = {"random_state": 42, "ground_truth_fraction": 0.2,
          "validation_fraction": 0.2, "reference_year": 2016}
display(pd.Series(config, name="diagnostic_configuration").to_frame())


,diagnostic_configuration
random_state,42.0
ground_truth_fraction,0.2
validation_fraction,0.2
reference_year,2016.0


In [3]:
from sklearn.model_selection import train_test_split

def prepare_data(config):
    """Assign stable source-row IDs, validate labels, deduplicate, then split."""
    raw = pd.read_csv(TRAIN_PATH)
    required = {"Brand", "Price", "Body", "Mileage", "EngineV", "Engine Type", "Registration", "Year", "Model"}
    missing = required - set(raw.columns)
    if missing:
        raise ValueError(f"Dataset is missing columns: {sorted(missing)}")
    if ID_COLUMN in raw:
        raise ValueError(f"Source column {ID_COLUMN} is reserved for generated row IDs")
    source_columns = raw.columns.tolist()
    raw.insert(0, ID_COLUMN, np.arange(1, len(raw) + 1))
    raw[TARGET] = pd.to_numeric(raw[TARGET], errors="coerce")
    valid_label = np.isfinite(raw[TARGET]) & (raw[TARGET] > 0)
    labeled = raw.loc[valid_label].copy()
    # Ignore generated IDs when identifying identical source records.
    duplicate = labeled.duplicated(subset=source_columns, keep="first")
    usable = labeled.loc[~duplicate].copy()
    if len(usable) < 10 or usable[TARGET].nunique() < 2:
        raise ValueError("Need at least ten labeled unique rows and two distinct positive prices")
    for key in ("ground_truth_fraction", "validation_fraction"):
        if not 0 < config[key] < 0.5:
            raise ValueError(f"{key} must be between 0 and 0.5")
    development, ground = train_test_split(usable, test_size=config["ground_truth_fraction"], random_state=config["random_state"])
    training, validation = train_test_split(development, test_size=config["validation_fraction"], random_state=config["random_state"])
    if config.get("cv_folds", 2) > len(training):
        raise ValueError("cv_folds exceeds training rows")
    assert not (set(training[ID_COLUMN]) & set(validation[ID_COLUMN])
                or set(training[ID_COLUMN]) & set(ground[ID_COLUMN])
                or set(validation[ID_COLUMN]) & set(ground[ID_COLUMN]))
    audit = {"source_rows": len(raw), "excluded_invalid_or_missing_prices": int((~valid_label).sum()),
             "excluded_exact_duplicates": int(duplicate.sum()), "usable_rows": len(usable),
             "training_rows": len(training), "validation_rows": len(validation), "ground_truth_rows": len(ground)}
    return training, validation, ground, audit


In [4]:
training, validation, ground, audit = prepare_data(config)
display(pd.Series(audit, name="data_preparation").to_frame())
print("Training, validation and ground-truth row IDs are disjoint.")
display(training.isna().sum().sort_values(ascending=False).to_frame("missing_count"))


,data_preparation
source_rows,4345
excluded_invalid_or_missing_prices,172
excluded_exact_duplicates,50
usable_rows,4123
training_rows,2638
validation_rows,660
ground_truth_rows,825


Training, validation and ground-truth row IDs are disjoint.


,missing_count
EngineV,96
RowId,0
Brand,0
Price,0
Body,0
Mileage,0
Engine Type,0
Registration,0
Year,0
Model,0


## Row-wise cleaning rules
Engine volumes outside (0, 6.5] are marked unknown under the notebook's dataset-specific assumption. Negative/nonfinite mileage and years outside [1886, reference_year] are also marked unknown. No price, mileage or year percentile deletion is performed. The reference year is 2016, the latest model year in this dataset, rather than the current calendar year.


In [5]:
def clean_features(X, reference_year):
    """Apply row-wise validity rules; fitted imputers handle remaining gaps."""
    data = X.drop(columns=[ID_COLUMN, TARGET, "Model"], errors="ignore").copy()
    for column in ("Mileage", "EngineV", "Year"):
        data[column] = pd.to_numeric(data[column], errors="coerce")
        data.loc[~np.isfinite(data[column]), column] = np.nan
    data.loc[data["Mileage"] < 0, "Mileage"] = np.nan
    # Original notebook's domain rule: values above 6.5 L are suspicious here.
    # Treat them as unknown; retain the car instead of deleting its record.
    data.loc[(data["EngineV"] <= 0) | (data["EngineV"] > 6.5), "EngineV"] = np.nan
    data.loc[(data["Year"] < 1886) | (data["Year"] > reference_year), "Year"] = np.nan
    for column in data.select_dtypes(exclude="number").columns:
        data[column] = data[column].map(lambda value: str(value).strip() if pd.notna(value) else np.nan)
        data[column] = data[column].replace("", np.nan)
    return data

def engineer_features(X, reference_year):
    """Keep domain features deterministic and independent of prices."""
    data = clean_features(X, reference_year)
    data["AgeAtSnapshot"] = reference_year - data["Year"]
    data["LogMileage"] = np.log1p(data["Mileage"])
    # Mileage units are preserved from the source; no unit conversion is assumed.
    data["MileagePerYear"] = data["Mileage"] / (data["AgeAtSnapshot"] + 1)
    data["EngineVolumeMissing"] = data["EngineV"].isna().astype(int)
    # Age and Year encode the same linear information; keep one representation.
    return data.drop(columns="Year")


In [6]:
X = training.drop(columns=TARGET)
cleaned = clean_features(X, config["reference_year"])
display(cleaned.head(10))
display(pd.DataFrame({"missing_before": X.isna().sum(), "missing_after_rules": cleaned.isna().sum()}).dropna(how="all"))
print("Suspicious training engine volumes:", int(((X.EngineV <= 0) | (X.EngineV > 6.5)).sum()))


,Brand,Body,Mileage,EngineV,Engine Type,Registration,Year
828,Toyota,hatch,92.0,1.3,Petrol,yes,2006.0
1250,Volkswagen,crossover,92.0,3.0,Diesel,yes,2011.0
1672,Toyota,sedan,165.0,3.5,Gas,yes,2008.0
1072,BMW,sedan,246.0,3.0,Diesel,yes,2003.0
813,BMW,crossover,82.0,3.0,Petrol,yes,2011.0
2951,BMW,sedan,214.0,1.9,Petrol,no,2000.0
963,BMW,crossover,0.0,2.0,Diesel,yes,2016.0
3151,Mercedes-Benz,sedan,330.0,2.0,Gas,yes,1988.0
3394,Mercedes-Benz,other,356.0,2.3,Petrol,yes,1991.0
1360,BMW,crossover,71.0,4.4,Petrol,yes,2011.0


,missing_before,missing_after_rules
Body,0,0.0
Brand,0,0.0
Engine Type,0,0.0
EngineV,96,113.0
Mileage,0,0.0
Model,0,NaN
Registration,0,0.0
RowId,0,NaN
Year,0,0.0


Suspicious training engine volumes: 17


## Training-fitted imputation, scaling and encoding
Numeric medians and categorical modes come from training rows. Linear models receive standardized numeric features; trees do not require scaling. OneHotEncoder handles unseen categories without hard-coded dummy columns. In actual training, these learned steps are fitted anew inside each CV fold.


In [7]:
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor

def make_preprocessor(X: pd.DataFrame, scale_numeric: bool = True) -> ColumnTransformer:
    numeric = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical = X.select_dtypes(exclude=[np.number]).columns.tolist()
    numeric_steps = [("impute", SimpleImputer(strategy="median", keep_empty_features=True))]
    if scale_numeric:
        numeric_steps.append(("scale", StandardScaler()))
    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:  # scikit-learn < 1.2
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
    return ColumnTransformer(
        transformers=[
            ("numeric", Pipeline(numeric_steps), numeric),
            ("categorical", Pipeline([
                ("impute", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),
                ("onehot", encoder),
            ]), categorical),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

def price_model(X, estimator):
    # sklearn's estimator composition keeps learned preprocessing inside CV.
    # X already contains stateless engineered features, with Id removed.
    trees = (GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor, DecisionTreeRegressor)
    regressor = Pipeline([
        ("preprocess", make_preprocessor(X, scale_numeric=not isinstance(estimator, trees))),
        ("model", estimator),
    ])
    return TransformedTargetRegressor(regressor=regressor, func=np.log1p, inverse_func=np.expm1)


In [8]:
engineered = engineer_features(X, config["reference_year"])
preprocessor = make_preprocessor(engineered)
matrix = preprocessor.fit_transform(engineered)
assert np.isfinite(matrix).all()
assert ID_COLUMN not in engineered and TARGET not in engineered and "Model" not in engineered
print("Finite diagnostic training matrix:", matrix.shape)


Finite diagnostic training matrix: (2638, 25)


**My conclusions:** document uncertainty in the engine-volume rule and propose any alternative cleaning policy as an experiment evaluated on training CV. This diagnostic fit does not create or replace the candidates used for testing.
